In [ ]:
import json
from datetime import date
import re
from urllib.parse import urlparse
import html

# Helpes

In [82]:
def first_str(value):
    v = None
    if isinstance(value, list):
        v = value[0].strip() if value else None
    if isinstance(value, str):
        v = value.strip()
    if v:
        v = html.unescape(v)
        return v
    return None

In [5]:
def normalize_doi(value):
    if not value:
        return None
    value = str(value).strip()
    value = value.replace("https://doi.org/", "").replace("http://doi.org/", "")
    value = value.replace("doi:", "").strip()
    return value.lower() or None

In [57]:
def _assertions_say_open_access(assertions) -> bool | None:
    """
    Procura em message['assertion'] sinais como 'open access'.
    """
    if not isinstance(assertions, list):
        return None

    saw_restrictive = False

    for item in assertions:
        if not isinstance(item, dict):
            continue

        label = str(item.get("label", "")).strip().lower()
        name = str(item.get("name", "")).strip().lower()
        value = str(item.get("value", "")).strip().lower()

        text = " ".join([label, name, value])

        if "open access" in text or "open-access" in text:
            return True

        if "subscription" in text or "restricted" in text or "embargo" in text:
            saw_restrictive = True

    if saw_restrictive:
        return False

    return None


In [75]:
def find_issn_by_type(issn_types, desired_type):
    if not isinstance(issn_types, list):
        return None
    for item in issn_types:
        if not isinstance(item, dict):
            continue
        if item.get("type") == desired_type:
            return item.get("value")
    return None

In [87]:
def clean_text(value):
    if value is None:
        return None
    value = str(value).strip()
    return value or None

In [89]:
def join_name(given, family):
    return " ".join([p for p in [given, family] if p]) or None

In [93]:
def normalize_orcid(value):
    if not value:
        return None
    value = str(value).strip()
    value = value.replace("https://orcid.org/", "").replace("http://orcid.org/", "")
    return value or None

In [102]:
def get_affiliation(item):
    
    aff = item.get("affiliation")
    if isinstance(aff, list):
        if len(aff) == 0:
            return None
        affiliations = []
        for a in aff:
            name = clean_text(a.get("name"))
            if name:
                affiliations.append(name)
        return affiliations if affiliations else None
    return None

In [125]:
def normalize_org_name(name):
    if not name:
        return None
    name = re.sub(r"\s+", " ", name).strip().lower()
    return name

# Parser

In [7]:
datas = []
with open('data/artigos/val.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        datas.append(json.loads(line))
data = datas[0]

persistência em ordem:
1. upsert do PublicationContainer
2. upsert do Publication
3. upsert dos Author
4. insert de PublicationContributor
5. upsert dos Funder
6. insert de PublicationFunder
7. insert de PublicationKeyword
8. insert de PublicationMetricSnapshot
9. insert de PublicationReference

In [ ]:
def parser_container(data):
    container = {
        "name": first_str(data.get("container-title")),
        "alternate_name": first_str(data.get("short-container-title")),
        "publisher": data.get("publisher"),
        "issn_print": find_issn_by_type(data.get("issn-type"), "print"),
        "issn_electronic": find_issn_by_type(data.get("issn-type"), "electronic"),
        "isbn": first_str(data.get("ISBN"))
        }
    return container
parser_container(data)

In [ ]:
def parser_publication(data):
    publication = {
        "publication_type": data.get("type"),
        "title": first_str(data.get("title")),
        "subtitle": first_str(data.get("subtitle")),
        "alternative_title": first_str(data.get("short-title")) or first_str(data.get("original-title")),
        "abstract": data.get("abstract"),
        "date_published": parse_date(
                data.get("published")
                or data.get("published-print")
                or data.get("issued")
            ),
        "language": data.get("language"),
        "subject": parse_subject(data.get("subject")),
        "doi": normalize_doi(data.get("DOI")),
        "isbn": first_str(data.get("ISBN")),
        "identifier": first_str(data.get("alternative-id")),
        "publisher": data.get("publisher"),
        "url": (deep_get_url(data)
                or data.get("URL")
            ),
        "license": first_license_url(data.get("license")),
        "conditions_of_access": build_conditions_of_access(data),
        "is_accessible_for_free": infer_free_access(data),
        "page_start": parse_page_range(data.get("page"))[0],
        "page_end": parse_page_range(data.get("page"))[1],
        "volume_number": data.get("volume"),
        "issue_number": data.get("issue"),
        "edition": data.get("edition-number") or data.get("special_numbering"),
        "source": "crossref",
        "raw_json": data,
    }
    return publication
parser_publication(data)

In [ ]:
def parser_contributor(data):
    contributors = []
    for item in data.get("author"):
        given = clean_text(item.get("given"))
        family = clean_text(item.get("family"))
        full_name = join_name(given, family)
        orcid = normalize_orcid(item.get("ORCID"))
        c = { "author": {
                "name": full_name,
                "given_name": given,
                "family_name": family,
                "orcid": orcid,
                "lattes_id": None,
                "is_inpa_researcher": None,
            },
            "contributor": {
                "role": "author",
                "position": item.get("sequence"),
                "raw_name": full_name,
                "raw_affiliation": get_affiliation(item),
            }}
        contributors.append(c)

    return contributors
parser_contributor(data)

In [ ]:
def parse_funder(data):
    funders = []
    for item in data.get("funder"):
        funder_doi = normalize_doi(item.get("DOI"))
        name = clean_text(item.get("name"))
        funder = {
                "name": name,
                "standard_name": normalize_org_name(name),
                "doi": funder_doi,
            }
        f = {"funder": funder}
        awards = item.get("award")
        if isinstance(awards, list):
            award = awards[0] 
            f['publication_funder'] = {
                "award_number": clean_text(award)
            }

        funders.append(f)
        

    return funders
parse_funder(data)

In [142]:
def extract_metric_snapshot(data):
    return {
        "citation_count": int(data.get("is-referenced-by-count")),
        "reference_count": int(data.get("reference-count")),
        "altmetric_score": None,
        "mendeley_readers": None,
        "tweets_count": None,
        "news_count": None,
        "blog_count": None,
        "policy_count": None,
        "patent_count": None,
        "source": "crossref",
}

In [ ]:
def parser_references(data):
    references = []
    for item in data.get("reference", []):
        title = clean_text(item.get("article-title"))
        if not title:
            continue
        
        r = {
            "doi": normalize_doi(item.get("DOI")),
            "title": title,
            "author": clean_text(item.get("author")),
            "journal_title": clean_text(item.get("journal-title")),
            "year": item.get("year"),
            "volume": clean_text(item.get("volume")),
            "issue": clean_text(item.get("issue")),
            "match_source": None,
            }
        references.append(r)
    return references

parser_references(data)

In [158]:
def normalize_crossref_data(data):
    normalized = {
        "publication": parser_publication(data),
        "container": parser_container(data),
        "contributors": parser_contributor(data),
        "funders": parse_funder(data),
        "references": parser_references(data),
        "metric_snapshot": extract_metric_snapshot(data),
    }
    return normalized
norm = normalize_crossref_data(data)

In [161]:
norm

{'publication': {'publication_type': 'journal-article',
  'title': 'Immunometabolic costs of parasitism under warming: Impaired mitochondrial function and thermal tolerance in an Amazonian fish',
  'subtitle': None,
  'alternative_title': None,
  'abstract': None,
  'date_published': datetime.date(2026, 1, 1),
  'language': 'en',
  'subject': None,
  'doi': '10.1016/j.fsi.2025.110959',
  'isbn': None,
  'identifier': 'S1050464825008484',
  'publisher': 'Elsevier BV',
  'url': 'https://linkinghub.elsevier.com/retrieve/pii/S1050464825008484',
  'license': 'https://www.elsevier.com/tdm/userlicense/1.0/',
  'conditions_of_access': 'Acesso ao conteúdo sujeito à licença de mineração/texto e dados do editor | Licença: https://www.elsevier.com/tdm/userlicense/1.0/ | Vigência da licença a partir de 2026-01-01 | Links de conteúdo identificados: text/plain, text/xml | URL principal no domínio linkinghub.elsevier.com | Informações adicionais: copyright © 2025 Elsevier Ltd. All rights are reserved,

In [164]:
with open('data/artigos/normalized.json', 'w', encoding='utf-8') as f:
        json.dump(norm, f, indent=4, ensure_ascii=False, default=str)


In [ ]:
def ingest_crossref_message(session, message: dict) -> Publication:
    norm = normalize_crossref_message(message)

    # 1. container
    container = None
    if norm["container"]:
        container = get_or_create_container(session, norm["container"])

    # 2. publication
    publication = upsert_publication(session, norm["publication"], container)

    # 3. contributors
    replace_publication_contributors(session, publication, norm["contributors"])

    # 4. funders
    replace_publication_funders(session, publication, norm["funders"])

    # 5. keywords
    replace_publication_keywords(session, publication, norm["keywords"])

    # 6. métricas
    create_metric_snapshot(session, publication, norm["metric_snapshot"])

    # 7. references
    replace_publication_references(session, publication, norm["references"])

    session.flush()
    return publication

'Immunometabolic costs of parasitism under warming: Impaired mitochondrial function and thermal tolerance in an Amazonian fish'

In [10]:
with open('data/artigo.json', 'w', encoding='utf-8') as f:
    json.dump(artigo, f, indent=4, ensure_ascii=False)

In [11]:
reference = artigo['reference']
len(reference)

63